# Student 5 — Text Analyst: Social Media Crime Report NLP

**Assignment:** AI for Engineers — Multimodal Crime / Incident Report Analyzer  
**Role:** Student 5 — Text Analyst  
**Pipeline:** Load tweets → Clean/preprocess → spaCy NER → HuggingFace sentiment → Topic classification → Export CSV

### Required Output Columns
`Text_ID | Crime_Type | Location_Entity | Sentiment | Topic | Severity_Label`

---

## Dataset
**CrimeReport** — 115 real crime-related tweets (JSON Lines format)  
File: `CrimeReport (1).txt`  
Source: Social media posts about crimes, arrests, incidents

In [ ]:
# ============================================================
# CELL 1 — Install all dependencies
# ============================================================
!pip install spacy transformers torch nltk pandas -q
!python -m spacy download en_core_web_sm -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('All packages installed.')

In [ ]:
# ============================================================
# CELL 2 — Load CrimeReport dataset
# Format: JSON Lines — one JSON object per line
# ============================================================
import json
import pandas as pd

DATA_PATH = 'CrimeReport (1).txt'

records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass

raw_df = pd.DataFrame([
    {
        'id':         r.get('id', ''),
        'text':       r.get('text', ''),
        'created_at': r.get('created_at', ''),
        'place':      (r.get('place') or {}).get('full_name', '') if isinstance(r.get('place'), dict) else '',
    }
    for r in records
])

print(f'Loaded {len(raw_df)} records')
print(raw_df[['text', 'place']].head(5).to_string())

In [ ]:
# ============================================================
# CELL 3 — Text preprocessing with NLTK
# Steps: remove noise → normalize → tokenize → remove stopwords
# ============================================================
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    """Remove URLs, mentions, hashtag symbols, special chars; normalize whitespace."""
    text = re.sub(r'http\S+', '', text)             # remove URLs
    text = re.sub(r'@\w+', '', text)                # remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)           # strip # but keep word
    text = re.sub(r'[^\w\s.,!?;:\'-]', ' ', text)  # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()        # normalize whitespace
    return text

def tokenize_and_normalize(text):
    """Tokenize, remove stopwords, lemmatize."""
    tokens = word_tokenize(text.lower())
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t.isalpha() and t not in stop_words]
    return tokens

raw_df['clean_text'] = raw_df['text'].apply(clean_text)
raw_df['tokens']     = raw_df['clean_text'].apply(tokenize_and_normalize)

print('Preprocessing complete.')
print(raw_df[['text', 'clean_text', 'tokens']].head(3).to_string())

In [ ]:
# ============================================================
# CELL 4 — spaCy Named Entity Recognition (NER)
# Extracts: people, locations, organizations, dates
# ============================================================
import spacy

nlp = spacy.load('en_core_web_sm')

def extract_entities(text):
    """Run spaCy NER; return dict of entity types."""
    doc = nlp(text)
    return {
        'persons': [e.text for e in doc.ents if e.label_ == 'PERSON'],
        'locations': [e.text for e in doc.ents if e.label_ in ('GPE', 'LOC', 'FAC')],
        'orgs':    [e.text for e in doc.ents if e.label_ == 'ORG'],
        'dates':   [e.text for e in doc.ents if e.label_ == 'DATE'],
    }

def get_location_entity(text, place_tag):
    """Prefer tweet place tag; fall back to first spaCy LOC entity."""
    if place_tag:
        return place_tag
    ents = extract_entities(text)
    return ents['locations'][0] if ents['locations'] else 'Unknown'

raw_df['ner']             = raw_df['clean_text'].apply(extract_entities)
raw_df['Location_Entity'] = raw_df.apply(
    lambda row: get_location_entity(row['clean_text'], row['place']), axis=1
)

print('NER complete. Sample locations:')
print(raw_df[['clean_text', 'Location_Entity']].head(10).to_string())

In [ ]:
# ============================================================
# CELL 5 — Sentiment Analysis (HuggingFace Transformers)
# Model: distilbert-base-uncased-finetuned-sst-2-english
# ============================================================
from transformers import pipeline

sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english'
)

def get_sentiment(text):
    """Returns Positive / Negative / Neutral."""
    result = sentiment_pipe(text[:512])[0]  # DistilBERT 512-token limit
    if result['score'] < 0.6:
        return 'Neutral'
    return 'Positive' if result['label'] == 'POSITIVE' else 'Negative'

raw_df['Sentiment'] = raw_df['clean_text'].apply(get_sentiment)

print('Sentiment analysis complete.')
print(raw_df['Sentiment'].value_counts())

In [ ]:
# ============================================================
# CELL 6 — Topic & Crime Type Classification
# Rule-based keyword matching (fast, interpretable)
# ============================================================

CRIME_MAP = [
    (['shooting', 'shot', 'gunshot', 'gunfire'],          'Shooting'),
    (['murder', 'homicide', 'killed', 'dead', 'fatally'], 'Homicide'),
    (['robbery', 'robbed', 'theft', 'stolen', 'burglary'],'Robbery / Theft'),
    (['assault', 'attack', 'beat', 'stab', 'stabbed'],    'Assault'),
    (['fire', 'arson', 'blaze', 'burning'],               'Fire / Arson'),
    (['drug', 'narcotic', 'cocaine', 'heroin', 'meth'],   'Drug Offense'),
    (['accident', 'crash', 'collision'],                  'Road Accident'),
    (['arrest', 'arrested', 'warrant', 'charged'],        'Arrest'),
]

TOPIC_MAP = [
    (['shooting', 'shot', 'gun', 'firearm', 'bullet'],               'Shooting'),
    (['murder', 'homicide', 'killing', 'killed', 'dead', 'victim'],  'Homicide / Murder'),
    (['robbery', 'theft', 'stolen', 'burglary', 'larceny'],          'Robbery / Theft'),
    (['assault', 'attack', 'fight', 'violence'],                      'Assault'),
    (['fire', 'arson', 'blaze'],                                      'Fire / Arson'),
    (['accident', 'crash', 'collision', 'vehicle'],                   'Road Accident'),
    (['drug', 'narcotics'],                                           'Drug-Related'),
    (['arrest', 'police', 'officer', 'suspect'],                      'Police Activity'),
    (['disturbance', 'protest', 'riot'],                              'Public Disturbance'),
]

SEVERITY_MAP = {
    'High':   ['shooting', 'murder', 'homicide', 'killed', 'dead', 'assault', 'fire', 'arson', 'attack'],
    'Medium': ['robbery', 'theft', 'arrest', 'suspect', 'drug', 'stolen', 'crash'],
}

def classify(text, mapping):
    text_l = text.lower()
    for keywords, label in mapping:
        if any(k in text_l for k in keywords):
            return label
    return 'General Crime'

def get_severity(text):
    text_l = text.lower()
    for level, words in SEVERITY_MAP.items():
        if any(w in text_l for w in words):
            return level
    return 'Low'

raw_df['Crime_Type']    = raw_df['clean_text'].apply(lambda t: classify(t, CRIME_MAP))
raw_df['Topic']         = raw_df['clean_text'].apply(lambda t: classify(t, TOPIC_MAP))
raw_df['Severity_Label']= raw_df['clean_text'].apply(get_severity)

print('Topic classification complete.')
print(raw_df['Topic'].value_counts())

In [ ]:
# ============================================================
# CELL 7 — Validate and save text_output.csv
# ============================================================

REQUIRED_COLUMNS = ['Text_ID', 'Crime_Type', 'Location_Entity', 'Sentiment', 'Topic', 'Severity_Label']

# Assign Text_ID
raw_df['Text_ID'] = [f'TXT_{i+1:03d}' for i in range(len(raw_df))]

text_df = raw_df[REQUIRED_COLUMNS].copy()

# Validate
assert list(text_df.columns) == REQUIRED_COLUMNS, f'Column mismatch: {list(text_df.columns)}'
assert len(text_df) > 0, 'DataFrame is empty!'

text_df.to_csv('text_output.csv', index=False)

print(f'Saved text_output.csv')
print(f'  Rows: {len(text_df)} | Columns: {REQUIRED_COLUMNS}')
print()
print(text_df.head(10).to_string(index=False))